In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

In [2]:
from pathlib import Path
import subprocess
import requests


# Video source: https://www.pexels.com/video/dog-eating-854132/
# License: CC0. Author: Coverr.
url = "https://videos.pexels.com/video-files/854132/854132-sd_640_360_25fps.mp4"
response=requests.get(url, headers={'User-Agent':''})
if response.status_code!=200: raise RuntimeError(f'Failed to download video. {response.status_code=}')

temp_dir=Path('D:/results/temp')
short_video_path=temp_dir/"short_video.mp4"
with open(short_video_path, 'wb') as f:
    for chunk in response.iter_content(): f.write(chunk)

long_video_path=temp_dir/'long_video.mp4'
ffmpeg_command=['ffmpeg', 
               '-stream_loop', '50', # repeat video 51 times to get ~12 mins
               '-i', f'{short_video_path}',
               '-c', 'copy',
                f'{long_video_path}'
]
subprocess.run(ffmpeg_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

from torchcodec.decoders import VideoDecoder
print(f'Short video duration: {VideoDecoder(short_video_path).metadata.duration_seconds} seconds')
print(f'Long video duration: {VideoDecoder(long_video_path).metadata.duration_seconds/60} mins')

Short video duration: 13.8 seconds
Long video duration: 11.5 mins


In [3]:
from pathlib import Path
import subprocess
import json
from time import perf_counter_ns

# Let's define a simple function to run ffprobe on a video's first stream index, then writes the results in output_json_path
def generate_frame_mappings(video_path, output_json_path, stream_index):
    ffprobe_cmd=["ffprobe",
                "-i", f"{video_path}",
                "-select_streams", f"{stream_index}",
                "-show_frames",
                "-show_entries",
                "frame=pts,duration,key_frame",
                "-of", "json"
    ]
    print(f"Running ffprobe:\n{' '.join(ffprobe_cmd)}\n")
    ffprobe_result=subprocess.run(ffprobe_cmd, check=True, capture_output=True, text=True)
    with open(output_json_path, 'w') as f: f.write(ffprobe_result.stdout)

stream_index=0
long_json_path=temp_dir/"long_custom_frame_mappings.json"
short_json_path=temp_dir/"short_custom_frame_mappings.json"

generate_frame_mappings(long_video_path, long_json_path, stream_index)
generate_frame_mappings(short_video_path, short_json_path, stream_index)
with open(short_json_path) as f: sample_data=json.loads(f.read())
print('Sample of fields in custom frame mappings: ')
for frame in sample_data['frames'][:3]:
    print(f"{frame['key_frame']=}, {frame['pts']=}, {frame['duration']=}")

Running ffprobe:
ffprobe -i D:\results\temp\long_video.mp4 -select_streams 0 -show_frames -show_entries frame=pts,duration,key_frame -of json

Running ffprobe:
ffprobe -i D:\results\temp\short_video.mp4 -select_streams 0 -show_frames -show_entries frame=pts,duration,key_frame -of json

Sample of fields in custom frame mappings: 
frame['key_frame']=1, frame['pts']=0, frame['duration']=1
frame['key_frame']=0, frame['pts']=1, frame['duration']=1
frame['key_frame']=0, frame['pts']=2, frame['duration']=1


In [4]:
print("type(sample_data): ", type(sample_data), " sample_data.keys(): ", sample_data.keys(), " type(sample_data['frames']): ", 
      type(sample_data['frames']), " len(sample_data['frames']) ", len(sample_data['frames']))
print("type(sample_data['frames'][0]): ", type(sample_data['frames'][0]), " sample_data['frames'][0].keys() ", 
     sample_data['frames'][0].keys())

type(sample_data):  <class 'dict'>  sample_data.keys():  dict_keys(['frames'])  type(sample_data['frames']):  <class 'list'>  len(sample_data['frames'])  345
type(sample_data['frames'][0]):  <class 'dict'>  sample_data['frames'][0].keys()  dict_keys(['key_frame', 'pts', 'duration', 'side_data_list'])


In [10]:
import torch


# Here, we define a benchmarking function, with the option to seek to the start of a file_like.
def bench(f, file_like=False, average_over=50, warmup=2, **f_kwargs):
    for _ in range(warmup):
        f(**f_kwargs)
        if file_like:
            f_kwargs["custom_frame_mappings"].seek(0)

    times = []
    for _ in range(average_over):
        start = perf_counter_ns()
        f(**f_kwargs)
        end = perf_counter_ns()
        times.append(end - start)
        if file_like:
            f_kwargs["custom_frame_mappings"].seek(0)

    times = torch.tensor(times) * 1e-6  # ns to ms
    std = times.std().item()
    med = times.median().item()
    print(f"{med = :.2f}ms +- {std:.2f}")


for video_path, json_path in ((short_video_path, short_json_path), (long_video_path, long_json_path)):
    print(f"\nRunning benchmarks on {Path(video_path).name}")

    print("Creating a VideoDecoder object with custom_frame_mappings:")
    with open(json_path, "r") as f:
        bench(VideoDecoder, file_like=True, source=video_path, stream_index=stream_index, custom_frame_mappings=f)

    # Compare against exact seek_mode
    print("Creating a VideoDecoder object with seek_mode='exact':")
    bench(VideoDecoder, source=video_path, stream_index=stream_index, seek_mode="exact")


Running benchmarks on short_video.mp4
Creating a VideoDecoder object with custom_frame_mappings:
med = 5.19ms +- 0.59
Creating a VideoDecoder object with seek_mode='exact':
med = 5.20ms +- 0.13

Running benchmarks on long_video.mp4
Creating a VideoDecoder object with custom_frame_mappings:
med = 30.37ms +- 0.95
Creating a VideoDecoder object with seek_mode='exact':
med = 42.29ms +- 1.39


In [11]:
def decode_frames(video_path, seek_mode='exact', custom_frame_mappings=None):
    decoder=VideoDecoder(source=video_path, seek_mode=seek_mode, custom_frame_mappings=custom_frame_mappings)
    decoder.get_frames_in_range(start=0, stop=10)

for video_path, json_path in ((short_video_path, short_json_path), (long_video_path, long_json_path)):
    print(f"\nRunning benchmarks on {Path(video_path).name}")
    print("Decoding frames with custom_frame_mappings: ")
    with open(json_path, 'r') as f:
        bench(decode_frames, file_like=True, video_path=video_path, custom_frame_mappings=f)
    print("Decoding frames with seek_mode=exact: ")
    bench(decode_frames, video_path=video_path, seek_mode='exact')


Running benchmarks on short_video.mp4
Decoding frames with custom_frame_mappings: 
med = 14.70ms +- 0.27
Decoding frames with seek_mode=exact: 
med = 14.70ms +- 0.22

Running benchmarks on long_video.mp4
Decoding frames with custom_frame_mappings: 
med = 41.44ms +- 1.39
Decoding frames with seek_mode=exact: 
med = 50.91ms +- 0.58


In [12]:
print("Metadata of short video with custom_frame_mappings: ")
with open(short_json_path, 'r') as f: print(VideoDecoder(short_video_path, custom_frame_mappings=f).metadata)
print("Metadata of short video with seek_mode=exact")
print(VideoDecoder(short_video_path, seek_mode='exact').metadata)

with open(short_json_path, 'r') as f:
    custom_frame_mappings_decoder=VideoDecoder(short_video_path, custom_frame_mappings=f)
exact_decoder=VideoDecoder(short_video_path, seek_mode='exact')
for i in range(len(exact_decoder)):
    torch.testing.assert_close(
        exact_decoder.get_frame_at(i).data,
        custom_frame_mappings_decoder.get_frame_at(i).data,
        atol=0, rtol=0
    )
print("Frame seeking is the same for this video!")

Metadata of short video with custom_frame_mappings: 
VideoStreamMetadata:
  duration_seconds_from_header: 13.8
  begin_stream_seconds_from_header: 0.0
  bit_rate: 505790.0
  codec: h264
  stream_index: 0
  duration_seconds: 13.8
  begin_stream_seconds: 0.0
  begin_stream_seconds_from_content: None
  end_stream_seconds_from_content: None
  width: 640
  height: 360
  num_frames_from_header: 345
  num_frames_from_content: None
  average_fps_from_header: 25.0
  pixel_aspect_ratio: 1
  end_stream_seconds: 13.8
  num_frames: 345
  average_fps: 25.0

Metadata of short video with seek_mode=exact
VideoStreamMetadata:
  duration_seconds_from_header: 13.8
  begin_stream_seconds_from_header: 0.0
  bit_rate: 505790.0
  codec: h264
  stream_index: 0
  duration_seconds: 13.8
  begin_stream_seconds: 0.0
  begin_stream_seconds_from_content: 0.0
  end_stream_seconds_from_content: 13.8
  width: 640
  height: 360
  num_frames_from_header: 345
  num_frames_from_content: 345
  average_fps_from_header: 25.0
